# Bank Customer Churn — CS289A Path 2
Colab experiments implementing the project proposal. No results are precomputed or claimed.

Run all cells in order. Default smoke mode trains four small models on 10% of the training split. Set `MODE="full"` and enable experiment flags for the required studies. Drive checkpoints allow interrupted runs to resume. Test evaluation requires a separate explicit switch after validation selection.

## 1. Environment Setup
Colab supplies these packages. If an import fails, install pandas, numpy, scikit-learn, matplotlib, joblib, and torch, then restart the runtime. Package versions are included in checkpoint identities; use the same versions and device across collaborators for closest reproducibility.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import copy, hashlib, io, json, platform, random, tempfile, time, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from IPython.display import display
print({"python": platform.python_version(), "sklearn": sklearn.__version__,
       "torch": str(torch.__version__), "numpy": np.__version__})

## 2. Google Drive and Dataset Loading

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Shared project folder (If it fails, please shortcut folders in MyDrive/CS189)
PROJECT_DIR = "/content/drive/MyDrive/CS189/CS289A_Graduate_Project"
DATA_PATH = os.path.join(PROJECT_DIR, "data", "train.csv")

# Check data access
assert os.path.exists(DATA_PATH), f"Dataset not found: {DATA_PATH}"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

## 3. Reproducibility and Configuration
Seed 42 controls splits, training subsets, estimators, weight initialization, shuffling, and dropout. Deterministic PyTorch kernels are requested; unsupported operations emit warnings. GPU/CPU arithmetic, library versions, hardware, and parallel reductions may still differ. Timing is inherently variable. Atomic replacement reduces partial-file risk on mounted Drive, but Drive synchronization is not a cross-user locking mechanism: avoid simultaneous writers to the same experiment.

`RUN_EXPERIMENTS` controls the expensive sections. Smoke runs are explicitly labeled and excluded from final model selection. The full baseline uses 100% of the training partition. No preprocessing or feature design uses test data.

In [ ]:
SEED = 42
MODE = "smoke"  # "smoke" or "full"
assert MODE in {"smoke", "full"}
RUN_EXPERIMENTS = {"hyperparameters": False, "features": False,
                   "data_size": False, "architecture": False}
RUN_FINAL_TEST = False  # enable only after reviewing and freezing validation selections
TUNE_THRESHOLD = False  # optional validation-only F1 threshold search
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = Path(PROJECT_DIR) / "models"
RESULT_DIR = Path(PROJECT_DIR) / "results"
for folder in (MODEL_DIR, RESULT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything()
def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), allow_nan=False)
def digest(obj):
    return hashlib.sha256(canonical(obj).encode()).hexdigest()
def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def atomic_write(path, writer):
    path = Path(path)
    fd, temporary = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(fd)
    try:
        writer(temporary)
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)
def save_json(path, obj):
    atomic_write(path, lambda p: Path(p).write_text(json.dumps(obj, indent=2)))
def save_csv(path, frame):
    atomic_write(path, lambda p: frame.to_csv(p, index=False))
def save_plot(name):
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
print("Mode:", MODE, "Device:", DEVICE)

## 4. Exploratory Data Analysis
First inspect schema, missing values, and duplicate counts. Feature distributions and correlations are drawn from training rows in section 5 to keep exploratory decisions away from the test partition. Repeated CustomerId values can occur in this synthetic dataset; the prescribed stratified row split measures row-level generalization, not unseen-customer generalization. We retain rows and exclude identifiers from predictors.

In [ ]:
TARGET = "Exited"
EXCLUDED = ["id", "CustomerId", "Surname"]
GROUPS = {"demographics": ["Geography", "Gender", "Age"],
          "financial": ["CreditScore", "Balance", "EstimatedSalary"],
          "account": ["Tenure", "NumOfProducts", "HasCrCard"],
          "activity": ["IsActiveMember"]}
FEATURES = [f for group in GROUPS.values() for f in group]
assert set(FEATURES + EXCLUDED + [TARGET]).issubset(df.columns)
assert df[TARGET].notna().all() and set(df[TARGET].unique()) == {0, 1}
print("Rows / columns:", df.shape, "(expected S4E1: 165034 / 14)")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate predictor + target rows:", df.duplicated(FEATURES + [TARGET]).sum())
print("Repeated CustomerId rows:", df.CustomerId.duplicated().sum())

## 5. Data Splitting and Preprocessing
One stratified 70/15/15 row split is saved as positional indices plus metadata under `results/`. A SHA-256 of the input CSV protects against changed content or row order. All experiments reuse these indices; data-size subsets are stratified and contained within the fixed training partition. Copy the split file and metadata with the dataset to another Drive, or regenerate with matching scikit-learn versions.

Numerical columns use training medians; categorical columns use the most frequent training value and one-hot encoding with unknown categories ignored. LR, SVM, and MLP standardize numerical features; RF leaves them unscaled. Each experiment fits its own preprocessor on its actual training subset.

In [ ]:
DATASET_ID = file_hash(DATA_PATH)
y = df[TARGET].astype(int)
SPLIT_SPEC = {"dataset": DATASET_ID, "seed": SEED, "ratios": [0.70, 0.15, 0.15],
              "algorithm": "stratified-two-stage-v1", "sklearn": sklearn.__version__}
SPLIT_ID = digest(SPLIT_SPEC)
split_path = RESULT_DIR / f"split_{SPLIT_ID}.npz"
if split_path.exists():
    with np.load(split_path, allow_pickle=False) as saved:
        train_idx, val_idx, test_idx = [saved[k] for k in ("train", "validation", "test")]
else:
    train_idx, held_idx = train_test_split(np.arange(len(df)), test_size=0.30,
                                         stratify=y, random_state=SEED)
    val_idx, test_idx = train_test_split(held_idx, test_size=0.50,
                                       stratify=y.iloc[held_idx], random_state=SEED)
    def write_split(p):
        with open(p, "wb") as stream:
            np.savez_compressed(stream, train=train_idx, validation=val_idx, test=test_idx)
    atomic_write(split_path, write_split)
save_json(split_path.with_suffix(".json"), SPLIT_SPEC)
joined = np.concatenate([train_idx, val_idx, test_idx])
assert np.array_equal(np.sort(joined), np.arange(len(df)))
SPLIT_CONTENT_ID = hashlib.sha256(joined.astype("int64").tobytes()).hexdigest()
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
display(y.iloc[train_idx].value_counts().rename_axis("Exited").to_frame("training_count"))
eda = df.iloc[train_idx]
eda[FEATURES].hist(figsize=(13, 9), bins=30)
save_plot("training_distributions")
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, feature in zip(axes, ["Geography", "Gender"]):
    eda[feature].value_counts(dropna=False).plot.bar(ax=ax, title=feature)
save_plot("training_categories")
corr = eda[FEATURES + [TARGET]].select_dtypes(include=np.number).corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax)
save_plot("training_correlations")

def subset_indices(fraction):
    if fraction == 1.0:
        return train_idx.copy()
    idx, _ = train_test_split(train_idx, train_size=fraction,
                              stratify=y.iloc[train_idx], random_state=SEED)
    return idx

def make_preprocessor(features, scale):
    categorical = [f for f in features if f in ["Geography", "Gender"]]
    numeric = [f for f in features if f not in categorical]
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("numeric", Pipeline(num_steps), numeric),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
          ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical)
    ], remainder="drop", sparse_threshold=0)

## 6. Shared Training and Evaluation Utilities
Each configuration includes data/split fingerprints, exact training subset, features, preprocessing version, package versions, device, seed, and hyperparameters. Changing implementation logic requires incrementing `IMPLEMENTATION_VERSION`. Only load trusted project checkpoints: joblib and PyTorch files are executable serialization formats.

Checkpoints store original fit time; reuse records load status separately. Inference timing includes preprocessing and prediction, excludes metrics and disk I/O, and synchronizes CUDA. SVM margins are used directly for AUC; they are never described as probabilities. Default thresholds are 0 for SVM and 0.5 for other models.

In [ ]:
IMPLEMENTATION_VERSION = "1"
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "sklearn": sklearn.__version__,
            "torch": str(torch.__version__), "joblib": joblib.__version__}
REGISTRY = {}

def make_config(model, hp, fraction=1.0, removed="none"):
    features = [f for f in FEATURES if f not in GROUPS.get(removed, [])]
    idx = subset_indices(fraction)
    return {"implementation": IMPLEMENTATION_VERSION, "model": model,
            "hp": copy.deepcopy(hp), "seed": SEED, "dataset": DATASET_ID,
            "split": SPLIT_ID, "split_content": SPLIT_CONTENT_ID,
            "subset": hashlib.sha256(idx.astype("int64").tobytes()).hexdigest(),
            "fraction": fraction, "removed": removed, "features": features,
            "preprocessing": {"version": 1, "numeric_imputer": "median",
                "categorical_imputer": "most_frequent", "encoding": "onehot-ignore-unknown",
                "scale_numeric": model != "rf"}, "versions": VERSIONS,
            "device": str(DEVICE) if model == "mlp" else "cpu"}

def make_estimator(model, hp):
    if model == "lr":
        return LogisticRegression(**hp, random_state=SEED, max_iter=1000)
    if model == "svm":
        return LinearSVC(**hp, random_state=SEED, max_iter=10000, dual=False)
    if model == "rf":
        return RandomForestClassifier(**hp, random_state=SEED, n_jobs=2)
    raise ValueError(model)

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def predict_scores(bundle, X):
    if bundle["config"]["model"] == "mlp":
        net = bundle["network"]
        data = bundle["preprocessor"].transform(X).astype("float32")
        scores = []
        net.eval()
        with torch.no_grad():
            for start in range(0, len(data), 4096):
                xb = torch.from_numpy(data[start:start+4096]).to(DEVICE)
                scores.append(torch.sigmoid(net(xb)).cpu().numpy())
        return np.concatenate(scores)
    pipe = bundle["pipeline"]
    if bundle["config"]["model"] == "svm":
        return pipe.decision_function(X)
    return pipe.predict_proba(X)[:, 1]

def evaluate(bundle, idx, threshold=None):
    if threshold is None:
        threshold = 0.0 if bundle["config"]["model"] == "svm" else 0.5
    X = df.iloc[idx][bundle["config"]["features"]]
    synchronize()
    start = time.perf_counter()
    scores = predict_scores(bundle, X)
    synchronize()
    elapsed = time.perf_counter() - start
    truth = y.iloc[idx].to_numpy()
    pred = (scores >= threshold).astype(int)
    metrics = {"accuracy": accuracy_score(truth, pred),
               "precision": precision_score(truth, pred, zero_division=0),
               "recall": recall_score(truth, pred, zero_division=0),
               "f1": f1_score(truth, pred, zero_division=0),
               "roc_auc": roc_auc_score(truth, scores),
               "confusion_matrix": json.dumps(confusion_matrix(truth, pred, labels=[0, 1]).tolist()),
               "inference_seconds": elapsed, "threshold": threshold}
    return metrics, scores

def upsert_csv(path, row, keys):
    new = pd.DataFrame([row])
    if path.exists():
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new = new.drop_duplicates(keys, keep="last")
    save_csv(path, new)

def train_or_load(config):
    exp_id = digest(config)
    path = MODEL_DIR / f"{config['model']}_{exp_id}.{'pt' if config['model']=='mlp' else 'joblib'}"
    loaded = False
    if path.exists():
        try:
            if config["model"] == "mlp":
                bundle = torch.load(path, map_location="cpu", weights_only=False)
            else:
                bundle = joblib.load(path)
            if bundle["config"] != config:
                raise ValueError("Checkpoint metadata mismatch")
            loaded = True
        except Exception as exc:
            warnings.warn(f"Cannot load {path.name}; retraining: {exc}")
    if not loaded:
        seed_everything()
        idx = subset_indices(config["fraction"])
        X = df.iloc[idx][config["features"]]
        truth = y.iloc[idx]
        synchronize()
        start = time.perf_counter()
        if config["model"] == "mlp":
            bundle = fit_mlp(config, X, truth)
        else:
            pipe = Pipeline([("preprocess", make_preprocessor(config["features"], config["model"] != "rf")),
                             ("model", make_estimator(config["model"], config["hp"]))])
            pipe.fit(X, truth)
            bundle = {"pipeline": pipe, "history": []}
        synchronize()
        bundle.update(config=config, training_seconds=time.perf_counter()-start)
        if config["model"] == "mlp":
            serial = {k: v for k, v in bundle.items() if k != "network"}
            atomic_write(path, lambda p: torch.save(serial, p))
        else:
            atomic_write(path, lambda p: joblib.dump(bundle, p))
    if config["model"] == "mlp" and "network" not in bundle:
        net = MLP(bundle["input_dim"], config["hp"]["hidden"], config["hp"]["dropout"]).to(DEVICE)
        net.load_state_dict(bundle["state_dict"])
        bundle["network"] = net.eval()
    if bundle["history"]:
        save_csv(RESULT_DIR / f"history_{exp_id}.csv", pd.DataFrame(bundle["history"]))
    save_json(MODEL_DIR / f"{config['model']}_{exp_id}.json", config)
    return bundle, path, loaded

def run_experiment(model, hp, study, fraction=1.0, removed="none"):
    config = make_config(model, hp, fraction, removed)
    exp_id = digest(config)
    bundle, path, loaded = train_or_load(config)
    metrics, _ = evaluate(bundle, val_idx)
    row = {"experiment_id": exp_id, "model": model, "hyperparameters": canonical(hp),
           "seed": SEED, "feature_group_configuration": removed, "training_fraction": fraction,
           "mode": MODE, "study": study, "dataset_id": DATASET_ID, "split_id": SPLIT_ID,
           "training_seconds": bundle["training_seconds"], "checkpoint_path": str(path),
           "loaded_checkpoint": loaded, **metrics}
    upsert_csv(RESULT_DIR / "validation_results.csv", row, ["experiment_id", "mode", "study"])
    REGISTRY[exp_id] = config
    print(model, study, exp_id[:10], "loaded" if loaded else "trained", "validation AUC:", round(metrics["roc_auc"], 4))
    return row

def run_candidates(model, candidates, study):
    return [run_experiment(model, hp, study, fraction=BASE_FRACTION) for hp in candidates]

## 7. Logistic Regression

In [ ]:
BASE_FRACTION = 0.10 if MODE == "smoke" else 1.0
DEFAULTS = {"lr": {"C": 1.0}, "svm": {"C": 0.1},
            "rf": {"n_estimators": 30 if MODE == "smoke" else 150,
                   "max_depth": 12, "min_samples_leaf": 2},
            "mlp": {"hidden": [64, 32], "dropout": 0.2, "lr": 0.001,
                    "batch_size": 512, "epochs": 3 if MODE == "smoke" else 40,
                    "patience": 2 if MODE == "smoke" else 6, "min_delta": 0.0001}}
lr_baseline = run_experiment("lr", DEFAULTS["lr"], "baseline", BASE_FRACTION)

## 8. Linear SVM
LinearSVC avoids the cost of kernel SVM training. Its decision margins support ranking and ROC-AUC, but are not calibrated churn probabilities.

In [ ]:
svm_baseline = run_experiment("svm", DEFAULTS["svm"], "baseline", BASE_FRACTION)

## 9. Random Forest
No numerical scaling is applied. Tree count and parallelism are bounded for free Colab runtimes.

In [ ]:
rf_baseline = run_experiment("rf", DEFAULTS["rf"], "baseline", BASE_FRACTION)

## 10. PyTorch MLP
A configurable sequence of Linear/ReLU/Dropout layers outputs one logit. Adam optimizes binary cross entropy via backpropagation. Validation loss selects the best epoch; training stops after patience epochs without an improvement of at least `min_delta`. State tensors are copied to CPU for portable persistence. Both training and validation losses are weighted by the number of examples.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout):
        super().__init__()
        layers = []
        previous = input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.Dropout(dropout)])
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.layers = nn.Sequential(*layers)
    def forward(self, x):
        return self.layers(x).squeeze(-1)

def fit_mlp(config, X, truth):
    hp = config["hp"]
    prep = make_preprocessor(config["features"], True)
    xt = prep.fit_transform(X).astype("float32")
    xv = prep.transform(df.iloc[val_idx][config["features"]]).astype("float32")
    train_data = TensorDataset(torch.from_numpy(xt), torch.tensor(truth.to_numpy(), dtype=torch.float32))
    val_data = TensorDataset(torch.from_numpy(xv), torch.tensor(y.iloc[val_idx].to_numpy(), dtype=torch.float32))
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_data, batch_size=hp["batch_size"], shuffle=True,
                              generator=generator, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=4096, num_workers=0)
    net = MLP(xt.shape[1], hp["hidden"], hp["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=hp["lr"])
    criterion = nn.BCEWithLogitsLoss()
    best_loss, stale, best_epoch = float("inf"), 0, 0
    history, best_state = [], None
    for epoch in range(1, hp["epochs"] + 1):
        net.train()
        train_total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(net(xb), yb)
            loss.backward()
            optimizer.step()
            train_total += loss.item() * len(yb)
        net.eval()
        val_total = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_total += criterion(net(xb), yb).item() * len(yb)
        val_loss = val_total / len(val_data)
        if not np.isfinite(val_loss):
            raise RuntimeError("Non-finite MLP validation loss")
        history.append({"epoch": epoch, "train_loss": train_total / len(train_data), "validation_loss": val_loss})
        # Always retain the actual minimum-loss checkpoint. min_delta only controls patience.
        improvement = best_loss - val_loss
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        stale = 0 if improvement > hp["min_delta"] else stale + 1
        if stale >= hp["patience"]:
            break
    net.load_state_dict(best_state)
    return {"network": net.eval(), "preprocessor": prep, "state_dict": best_state,
            "input_dim": xt.shape[1], "best_epoch": best_epoch,
            "best_validation_loss": best_loss, "history": history}

mlp_baseline = run_experiment("mlp", DEFAULTS["mlp"], "baseline", BASE_FRACTION)

## 11. Hyperparameter Experiments
Small manual candidate lists replace a costly Cartesian grid. Only validation ROC-AUC selects configurations, with F1 as a tie breaker. Full runs reuse the 100% baseline. Smoke-only search results cannot enter final selection.

In [ ]:
if RUN_EXPERIMENTS["hyperparameters"]:
    run_candidates("lr", [{"C": c} for c in [0.1, 1.0, 10.0]], "hyperparameters")
    run_candidates("svm", [{"C": c} for c in [0.01, 0.1, 1.0]], "hyperparameters")
    run_candidates("rf", [DEFAULTS["rf"],
        {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 4},
        {"n_estimators": 100, "max_depth": 16, "min_samples_leaf": 8}], "hyperparameters")
    run_candidates("mlp", [DEFAULTS["mlp"],
        {**DEFAULTS["mlp"], "lr": 0.0005, "batch_size": 256},
        {**DEFAULTS["mlp"], "lr": 0.002, "batch_size": 1024}], "hyperparameters")

def validation_table():
    table = pd.read_csv(RESULT_DIR / "validation_results.csv")
    return table[(table.dataset_id == DATASET_ID) & (table.split_id == SPLIT_ID) &
                 table.experiment_id.isin(REGISTRY)].copy()

def select_configs():
    table = validation_table()
    eligible = table[(table["mode"] == "full") & (table.training_fraction == 1.0) &
                     (table.feature_group_configuration == "none") &
                     table.study.isin(["baseline", "hyperparameters", "architecture"])]
    selected = eligible.sort_values(["roc_auc", "f1", "experiment_id"], ascending=[False, False, True]).drop_duplicates("model")
    return {row.model: REGISTRY[row.experiment_id] for row in selected.itertuples()}

def ablation_configs():
    selected = select_configs()
    return {model: selected.get(model, make_config(model, hp, BASE_FRACTION))
            for model, hp in DEFAULTS.items()}